# GLDS-525 Pipeline, Notebook 4: Differential Gene Expression Analysis

**Study:** RR-23 Mouse Cerebellum Bulk RNA-Seq (NASA GeneLab OSD-525 / GLDS-525)  
**Groups:** Space Flight (FLT) vs Ground Control (GC) vs Vivarium Control (VIV)

---

## 🎓 Tutorial Introduction: What is Differential Gene Expression?

This is the final and most exciting notebook. After three notebooks of data
processing, we now answer the scientific question: **which genes change their
expression in response to spaceflight?**

### What does "differentially expressed" mean?
A gene is **differentially expressed (DE)** between two conditions if its
expression level is statistically significantly different. For example, if
a gene produces 500 reads in spaceflight mice but only 50 reads in ground
controls, it is "upregulated" during spaceflight.

### Why can't we just compare raw counts?
Raw counts are affected by:
- **Sequencing depth**: A sample sequenced to 30 million reads naturally gets
  more counts than one at 10 million, even if expression is identical
- **Gene length**: Longer genes naturally get more reads
- **Biological variation**: Mice are individuals, and expression differs even between
  animals in the same condition

**DESeq2** handles all of this using a negative binomial model that properly
accounts for count data properties, normalizes for sequencing depth, estimates
per-gene dispersion (biological variability), and applies statistical tests.

### Why three comparisons?
This study has three groups:
- **FLT** (Space Flight): mice aboard the ISS
- **GC** (Ground Control): mice in identical hardware on Earth, in a vivarium
- **VIV** (Vivarium Control): mice in standard vivarium housing

Comparing FLT vs GC isolates spaceflight effects. Comparing GC vs VIV tells
us if the special hardware itself affects gene expression. Comparing FLT vs VIV
shows total spaceflight vs normal conditions effects.

---

## What This Notebook Does

| Step | Description |
|---|---|
| **1** | Load config and count matrices |
| **2** | Load sample metadata and align to counts |
| **3** | Pre-filter low-count genes |
| **4** | Run DESeq2 via PyDESeq2 (all three contrasts) |
| **5** | Volcano plots: visualize DEGs |
| **6** | PCA: visualize sample clustering |
| **7** | Heatmaps: visualize top DEG expression patterns |
| **8** | Cross-validate against GeneLab pre-computed results |
| **9** | ORA: Over-Representation Analysis (pathway enrichment) |
| **10** | GSEA: Gene Set Enrichment Analysis |
| **11** | Summary DEG count table |

---

## Environment Setup

This notebook uses a **separate conda environment** from Notebooks 1–3:
```bash
conda create -n glds525_deg python=3.10
conda activate glds525_deg
pip install jupyter ipykernel pandas numpy matplotlib seaborn \
            pydeseq2 scipy statsmodels openpyxl adjustText gseapy scikit-learn
python -m ipykernel install --user --name glds525_deg --display-name "GLDS-525 DEG"
```

> 💡 **VS Code users:** After running `python -m ipykernel install` above, reload VS Code
> and select the **"GLDS-525 DEG"** kernel in the top-right kernel picker.

> **Input flexibility:** This notebook can use either:
> - Count matrices produced by **Notebook 3** (our pipeline), or
> - The count matrix **downloaded from GeneLab** directly
> 
> Set `USE_OUR_COUNTS = True` or `False` in Section 1 accordingly.


---
## Section 1: Load Configuration and Count Matrices

> **Overview:** We reload the pipeline config and select which count
> matrix to use. If you ran Notebooks 1–3, set `USE_OUR_COUNTS = True`. If you
> only want to run the DEG analysis using GeneLab's provided count table (without
> running the upstream pipeline yourself), set it to `False` and place the
> GeneLab count file in your `counts/` directory.


In [ ]:
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ProcessPoolExecutor
import gseapy as gp
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ── Load config ───────────────────────────────────────────────────────────────
CONFIG_PATH = Path.home() / 'GLDS525_project' / 'pipeline_config.json'
with open(CONFIG_PATH) as f:
    config = json.load(f)

PROJECT_ROOT = Path(config['PROJECT_ROOT'])
N_CORES      = config['N_CORES']
DIRS         = {k: Path(v) for k, v in config.items()
                if k not in ('PROJECT_ROOT', 'N_CORES', 'TOTAL_RAM_GB')}

# ══════════════════════════════════════════════════════════════════
# ← EDIT: True = use our Notebook 3 output; False = use GeneLab file
USE_OUR_COUNTS = True
# ══════════════════════════════════════════════════════════════════

if USE_OUR_COUNTS:
    COUNTS_FILE = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_merged_GLbulkRNAseq.csv'
    print("Using: OUR count matrix (Notebook 3 output)")
else:
    # Place the GeneLab file in the counts directory
    COUNTS_FILE = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_GLbulkRNAseq.csv'
    print("Using: GeneLab reference count matrix")

SAMPLES_FILE  = DIRS['runsheet'] / 'GLDS-525_rna_seq_SampleTable_GLbulkRNAseq.csv'
GL_DGE_FILE   = DIRS['counts']   / 'GLDS-525_rna_seq_differential_expression_rRNArm_GLbulkRNAseq.csv'

# Group settings
GROUP_LABELS = {'Space.Flight': 'FLT', 'Ground.Control': 'GC', 'Vivarium.Control': 'VIV'}
GROUP_COLORS = {'FLT': '#e74c3c', 'GC': '#3498db', 'VIV': '#2ecc71'}

print(f"\n✅ Config loaded. Cores: {N_CORES}")


In [ ]:
# Load count matrix
if not COUNTS_FILE.exists():
    raise FileNotFoundError(f"Count file not found: {COUNTS_FILE}\n"
                             "Run Notebook 3 first, or set USE_OUR_COUNTS=False "
                             "and place the GeneLab file in the counts directory.")

counts_raw = pd.read_csv(COUNTS_FILE, index_col=0)
print(f"Count matrix loaded: {counts_raw.shape}  (genes × samples)")
print(f"\nFirst 3 genes, first 4 columns:")
display(counts_raw.iloc[:3, :4])


---
## Section 2: Load Sample Metadata

> **Overview:** DESeq2 needs to know which experimental group each
> sample belongs to. This information comes from the **SampleTable**, a CSV file
> from GeneLab that maps sample names to conditions (Space Flight, Ground Control,
> Vivarium Control).
>
> We create a short abbreviation for each group (FLT, GC, VIV) and check that all
> samples in the count matrix have matching metadata entries.


In [ ]:
# Load SampleTable (from GeneLab)
samples_raw = pd.read_csv(SAMPLES_FILE, index_col=0)
print(f"SampleTable: {samples_raw.shape}")
print(f"Columns: {samples_raw.columns.tolist()}")

# Map condition labels to short group codes
samples = samples_raw.copy()
samples['group'] = samples['condition'].map(GROUP_LABELS)

if samples['group'].isna().any():
    print("\n⚠️  Unmapped condition values:")
    print(samples[samples['group'].isna()]['condition'].unique())
    print("   Update GROUP_LABELS dict in Section 1.")
else:
    print("\n✅ Group mapping successful:")
    print(samples['group'].value_counts().to_string())


In [ ]:
# Align sample names between count matrix and metadata
if counts_raw.shape[0] > counts_raw.shape[1]:
    counts_T = counts_raw.T   # rows=samples, cols=genes
else:
    counts_T = counts_raw

# Merge technical replicates if not already done
import re
def merge_techreps(df):
    pat = re.compile(r'(.+)_techrep\d+$')
    base_map = {}
    for col in df.index:
        m = pat.match(col)
        if m:
            base_map.setdefault(m.group(1), []).append(col)
    if not base_map:
        return df
    result = {}
    processed = set()
    for col in df.index:
        m = pat.match(col)
        if m:
            base = m.group(1)
            if base not in processed:
                result[base] = df.loc[base_map[base]].sum(axis=0)
                processed.add(base)
        else:
            result[col] = df.loc[col]
    return pd.DataFrame(result).T

counts_T = merge_techreps(counts_T)

# Find common samples
common = counts_T.index.intersection(samples.index)
only_counts = set(counts_T.index) - set(samples.index)
only_meta   = set(samples.index)  - set(counts_T.index)

print(f"Common samples:            {len(common)}")
print(f"In counts only (dropped):  {sorted(only_counts)}")
print(f"In metadata only (dropped):{sorted(only_meta)}")

common_sorted   = sorted(common)
counts_aligned  = counts_T.loc[common_sorted]      # samples × genes
samples_aligned = samples.loc[common_sorted]

print(f"\nFinal: {counts_aligned.shape[0]} samples × {counts_aligned.shape[1]} genes")


---
## Section 3: Pre-Filter Low-Count Genes

> **Overview:** Not all ~55,000 genes in the mouse genome are expressed
> in every tissue. In the cerebellum, many genes will have zero or near-zero counts
> across all samples. Including these genes:
> - Increases the number of statistical tests (worsening the multiple testing correction)
> - Uses more memory and computation time
> - Adds no information (you can't detect differential expression in a gene with no counts)
>
> **GeneLab's filtering criterion:** A gene must have at least **10 total counts**
> summed across all samples to be kept. This is a lenient threshold that removes
> only completely unexpressed genes while keeping genes with low-but-real expression.
>
> After filtering, we typically go from ~55,000 genes to ~20,000–25,000 expressed genes.


In [ ]:
# Convert to integer counts
counts_int = counts_aligned.round().astype(int)

# GeneLab filter: sum across all samples >= 10
SUM_THRESHOLD = 10
sum_filter    = counts_int.sum(axis=0) >= SUM_THRESHOLD

counts_filtered = counts_int.loc[:, sum_filter]

print(f"Filtering parameters (matching GeneLab):")
print(f"  Sum threshold:     {SUM_THRESHOLD} counts across all samples")
print(f"\nGenes before filter: {counts_int.shape[1]:,}")
print(f"Genes after filter:  {counts_filtered.shape[1]:,}")
print(f"Genes removed:       {counts_int.shape[1] - counts_filtered.shape[1]:,}")
print(f"\nFinal matrix: {counts_filtered.shape[0]} samples × {counts_filtered.shape[1]:,} genes")


---
## Section 4: Run DESeq2 (PyDESeq2)

> **Overview:** **DESeq2** is the gold-standard method for RNA-seq
> differential expression analysis. Here's what it does under the hood:
>
> 1. **Size factor normalization**: Estimates a scaling factor for each sample
>    to correct for differences in sequencing depth. A sample with twice as many
>    reads gets a size factor of ~2.
>
> 2. **Dispersion estimation**: For each gene, estimates how much biological
>    variability there is across samples in the same group. Genes with high
>    variability need stronger evidence before being called "significantly different."
>
> 3. **Negative binomial model**: Fits a statistical model to each gene that
>    accounts for both the count nature of the data and the estimated dispersion.
>
> 4. **Wald test**: Tests whether the fitted model shows a significant difference
>    between conditions.
>
> 5. **Benjamini-Hochberg correction**: Adjusts p-values for the fact that we're
>    testing ~20,000 genes simultaneously. The corrected values are called
>    **adjusted p-values (padj)** or **FDR (False Discovery Rate)**.
>
> We use **PyDESeq2**, a Python reimplementation of the original R DESeq2 package
> that produces nearly identical results.
>
> **Interpreting DESeq2 output:**
> - `log2FoldChange`: log₂(FLT/GC); positive means higher in FLT, negative means lower
> - `padj`: adjusted p-value; we use `padj < 0.05` as our significance threshold
> - `|log2FoldChange| > 1`: means at least 2-fold change between groups

> This step takes approximately **5–15 minutes** depending on your machine.


In [ ]:
import time
print("Fitting DESeq2 model...")
print(f"Input: {counts_filtered.shape[0]} samples × {counts_filtered.shape[1]:,} genes")
print(f"Using {N_CORES} CPU cores\n")

t0 = time.time()
dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=samples_aligned[['group']],
    design_factors='group',
    refit_cooks=True,
    n_cpus=N_CORES
)
dds.deseq2()
elapsed = (time.time() - t0) / 60
print(f"\n✅ DESeq2 complete ({elapsed:.1f} minutes)")


In [ ]:
# Define all three contrasts
CONTRASTS = [
    ('FLT', 'GC',  'Space Flight vs Ground Control'),
    ('FLT', 'VIV', 'Space Flight vs Vivarium Control'),
    ('GC',  'VIV', 'Ground Control vs Vivarium Control'),
]

results = {}

for cond1, cond2, label in CONTRASTS:
    key = f"{cond1}_vs_{cond2}"
    print(f"\nContrast: {label}")
    
    stat = DeseqStats(dds, contrast=['group', cond1, cond2], alpha=0.05)
    stat.summary()
    
    res = stat.results_df.copy()
    res['contrast']    = label
    res['significant'] = (res['padj'] < 0.05) & (res['log2FoldChange'].abs() > 1)
    results[key]       = res
    
    # Save results CSV
    out_path = DIRS['counts'] / f"DEGs_{key}_GLbulkRNAseq.csv"
    res.to_csv(out_path)
    
    n_up = ((res['padj'] < 0.05) & (res['log2FoldChange'] >  1)).sum()
    n_dn = ((res['padj'] < 0.05) & (res['log2FoldChange'] < -1)).sum()
    print(f"  DEGs (padj<0.05, |LFC|>1): {n_up + n_dn}  (↑{n_up} up, ↓{n_dn} down)")
    print(f"  Saved: {out_path.name}")

print("\n✅ All contrasts complete.")


---
## Section 5: Visualization: Volcano Plots

> **Overview:** A **volcano plot** is one of the most common visualizations
> in RNA-seq analysis. Each dot represents one gene, plotted as:
> - **X-axis**: log₂ fold change (how much the gene changed)
> - **Y-axis**: −log₁₀(adjusted p-value) (how statistically confident we are)
>
> The "volcano" shape emerges because:
> - Most genes cluster near the center (small change, low significance)
> - Significant DEGs appear in the upper-left (strongly downregulated) and
>   upper-right (strongly upregulated) "eruptions"
>
> **Thresholds used (matching standard practice):**
> - Vertical dashed lines: |log₂FC| > 1 (2-fold change)
> - Horizontal dashed line: padj < 0.05
>
> Red dots = upregulated in the first group, blue = downregulated, grey = not significant.
> Top 10 most significant DEGs are labeled with their gene name.


In [ ]:
def volcano_plot(res, title, ax, lfc_thresh=1.0, padj_thresh=0.05, n_label=10):
    df = res.dropna(subset=['padj', 'log2FoldChange']).copy()
    df['neg_log10_padj'] = -np.log10(df['padj'].clip(lower=1e-300))
    
    up = (df['padj'] < padj_thresh) & (df['log2FoldChange'] >  lfc_thresh)
    dn = (df['padj'] < padj_thresh) & (df['log2FoldChange'] < -lfc_thresh)
    ns = ~(up | dn)
    
    ax.scatter(df.loc[ns, 'log2FoldChange'], df.loc[ns, 'neg_log10_padj'],
               c='#cccccc', s=6, alpha=0.5, label='NS', rasterized=True)
    ax.scatter(df.loc[up, 'log2FoldChange'], df.loc[up, 'neg_log10_padj'],
               c='#e74c3c', s=10, alpha=0.85, label=f'Up ({up.sum()})')
    ax.scatter(df.loc[dn, 'log2FoldChange'], df.loc[dn, 'neg_log10_padj'],
               c='#3498db', s=10, alpha=0.85, label=f'Down ({dn.sum()})')
    
    ax.axvline(x= lfc_thresh, ls='--', lw=0.8, color='black')
    ax.axvline(x=-lfc_thresh, ls='--', lw=0.8, color='black')
    ax.axhline(y=-np.log10(padj_thresh), ls='--', lw=0.8, color='blue')
    
    top = df[up | dn].nsmallest(n_label, 'padj')
    for gene, row in top.iterrows():
        label_text = gene.split('_')[-1] if '_' in gene else gene[:12]
        ax.annotate(label_text, xy=(row['log2FoldChange'], row['neg_log10_padj']),
                    fontsize=6, ha='center', va='bottom',
                    xytext=(0, 3), textcoords='offset points')
    
    ax.set_xlabel('log2 Fold Change', fontsize=11)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')


fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (key, (c1, c2, label)) in zip(axes, zip(results.keys(), CONTRASTS)):
    volcano_plot(results[key], label, ax)

fig.suptitle('GLDS-525 RR-23 Mouse Cerebellum: Volcano Plots (|LFC|>1, padj<0.05)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = DIRS['counts'] / 'volcano_plots_GLDS525.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {out_path.name}")


---
## Section 6: Visualization: PCA

> **Overview:** **Principal Component Analysis (PCA)** reduces the
> high-dimensional gene expression data (thousands of genes) into just 2–3 dimensions
> that capture the most variation, so we can visualize all samples in a scatter plot.
>
> **What to look for:**
> - Samples from the same group (FLT, GC, VIV) should cluster together
> - Groups should be separated from each other, especially FLT vs GC
> - Outlier samples (isolated from their group) may have technical problems
>
> We use **VST (Variance-Stabilizing Transformation)** normalized counts for PCA,
> because raw counts violate the equal-variance assumption that PCA assumes.
>
> The percentage of variance explained by each PC tells you how much information
> each axis captures. PC1 usually captures the biggest biological difference
> (spaceflight vs ground).


In [ ]:
# Get VST-normalized counts from DESeq2 object
try:
    from pydeseq2.preprocessing import vst_transformation
    vst_counts = vst_transformation(dds)
    vst_df = pd.DataFrame(vst_counts, index=counts_filtered.index,
                          columns=counts_filtered.columns)
    print("VST computed from DESeq2 model")
except Exception:
    # Fallback: use log1p of normalized counts
    size_factors = dds.obsm.get('size_factors',
                    pd.Series(1.0, index=counts_filtered.index))
    norm_counts = counts_filtered.div(size_factors, axis=0)
    vst_df = np.log1p(norm_counts)
    print("Using log1p normalized counts (VST fallback)")

# Top 500 most variable genes
gene_var = vst_df.var(axis=0)
top500   = gene_var.nlargest(500).index
vst_top  = vst_df[top500]

# PCA
pca    = PCA(n_components=5)
pcs    = pca.fit_transform(StandardScaler().fit_transform(vst_top))
var_exp= pca.explained_variance_ratio_ * 100

pca_df = pd.DataFrame(pcs[:, :3], columns=['PC1','PC2','PC3'],
                      index=vst_top.index)
pca_df['group'] = samples_aligned.loc[pca_df.index, 'group'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (xpc, ypc) in zip(axes, [('PC1','PC2'), ('PC1','PC3')]):
    for grp, sub in pca_df.groupby('group'):
        ax.scatter(sub[xpc], sub[ypc], label=grp,
                   color=GROUP_COLORS[grp], s=90, alpha=0.88,
                   edgecolors='white', linewidths=0.5)
        for idx, row in sub.iterrows():
            ax.annotate(idx.replace('RR23_Cb_', ''),
                        (row[xpc], row[ypc]), fontsize=5.5,
                        ha='left', va='bottom',
                        xytext=(3, 2), textcoords='offset points')
    xi = int(xpc[-1]) - 1
    yi = int(ypc[-1]) - 1
    ax.set_xlabel(f"{xpc} ({var_exp[xi]:.1f}% variance)", fontsize=11)
    ax.set_ylabel(f"{ypc} ({var_exp[yi]:.1f}% variance)", fontsize=11)
    ax.legend(title='Group', fontsize=9)
    ax.set_title(f"{xpc} vs {ypc}", fontsize=12, fontweight='bold')

fig.suptitle('GLDS-525: PCA (top 500 variable genes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = DIRS['counts'] / 'PCA_GLDS525.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {out_path.name}")


---
## Section 7: Visualization: Heatmaps

> **Overview:** A **heatmap** shows the expression of many genes
> across all samples simultaneously as a color grid. We show the **top 50 DEGs**
> for each contrast, with:
> - Each row = one gene
> - Each column = one sample
> - Color = **z-score** of expression (how many standard deviations above/below the
>   mean for that gene across all samples)
>
> The **z-score normalization** removes differences in absolute expression levels
> between genes, so we can compare expression patterns across genes of very different
> scales on the same color scale.
>
> Rows are **clustered** (similar expression patterns grouped together). Columns are
> ordered by group (FLT, GC, VIV) so you can see group-level patterns.
>
> A good spaceflight heatmap should show a clear block of red (high expression) for
> FLT that's blue (low expression) in GC/VIV, or vice versa.


In [ ]:
def deg_heatmap(res, vst_df, samples_df, title, n_genes=50, filename=None):
    sig = res.dropna(subset=['padj']).query("padj < 0.05 and log2FoldChange.abs() > 1")
    if len(sig) == 0:
        print(f"No significant DEGs for: {title}")
        return
    
    top_genes = sig.nlargest(min(n_genes, len(sig)), 'stat').index
    top_genes = [g for g in top_genes if g in vst_df.columns]
    if not top_genes:
        print(f"No matching genes in VST matrix for: {title}")
        return
    
    heatmap_data = vst_df[top_genes].T
    heatmap_z    = heatmap_data.apply(zscore, axis=1)
    
    # Order columns by group
    col_order = []
    for g in ['FLT', 'GC', 'VIV']:
        col_order.extend([c for c in samples_df[samples_df['group']==g].index
                          if c in heatmap_z.columns])
    heatmap_z = heatmap_z[col_order]
    
    col_colors = pd.Series(
        [GROUP_COLORS[samples_df.loc[c, 'group']] for c in col_order],
        index=col_order
    )
    
    # Shorten gene labels (show gene name after _ if present)
    row_labels = [g.split('_')[-1] if '_' in g else g[:15]
                  for g in heatmap_z.index]
    heatmap_z.index = row_labels
    
    g = sns.clustermap(
        heatmap_z, col_cluster=False, col_colors=col_colors,
        cmap='RdBu_r', center=0,
        figsize=(14, max(10, len(top_genes) * 0.22)),
        yticklabels=True, xticklabels=True,
        linewidths=0.0, cbar_kws={'label': 'z-score'}
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(),
                                  rotation=45, ha='right', fontsize=7)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
    g.fig.suptitle(f"{title}: top {len(top_genes)} DEGs (z-scored VST)",
                   fontsize=12, fontweight='bold', y=1.01)
    patches = [mpatches.Patch(color=v, label=k) for k, v in GROUP_COLORS.items()]
    g.ax_col_dendrogram.legend(handles=patches, loc='center', ncol=3,
                                fontsize=9, title='Group')
    if filename:
        g.fig.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"✅ Saved: {Path(filename).name}")
    plt.show()


for key, (c1, c2, label) in zip(results.keys(), CONTRASTS):
    deg_heatmap(results[key], vst_df, samples_aligned, label, n_genes=50,
                filename=str(DIRS['counts'] / f"heatmap_{key}.png"))


---
## Section 8: Cross-Validate Against GeneLab Pre-Computed Results

> **Overview:** GeneLab has already run their own DESeq2 analysis and
> deposited the results. We compare our log₂ fold changes to theirs. If our pipeline
> matches GeneLab's (same parameters, same filtering), the correlation should be
> essentially perfect (r > 0.99).
>
> **Why might they differ slightly?**
> - PyDESeq2 vs R DESeq2 use slightly different numerical implementations
> - Different software versions can produce small rounding differences
> - These are not biologically meaningful; the gene lists will be nearly identical
>
> Download GeneLab's DGE file from the same accession page as the count tables.


In [ ]:
if GL_DGE_FILE.exists():
    gl_dge = pd.read_csv(GL_DGE_FILE, index_col=0)
    print(f"GeneLab DGE file: {gl_dge.shape}")
    print("\nAll columns:")
    for c in gl_dge.columns:
        print(f"  {c}")
else:
    print(f"⚠️  GeneLab DGE file not found: {GL_DGE_FILE}")
    print("   Download from GeneLab and place in the counts directory to run validation.")


In [ ]:
if GL_DGE_FILE.exists():
    lfc_cols  = [c for c in gl_dge.columns if 'log2fc' in c.lower() or 'lfc' in c.lower()]
    padj_cols = [c for c in gl_dge.columns if 'adj' in c.lower() and 'p' in c.lower()]
    
    print("LFC columns found:",  lfc_cols[:5])
    print("padj columns found:", padj_cols[:5])
    
    flt_gc_lfc = next((c for c in lfc_cols
                       if 'flight' in c.lower() or 'flt' in c.lower()), None)
    if flt_gc_lfc:
        print(f"\nAuto-detected FLT vs GC LFC column: {flt_gc_lfc}")
        
        merged = results['FLT_vs_GC'][['log2FoldChange']].join(
            gl_dge[[flt_gc_lfc]].rename(columns={flt_gc_lfc: 'GL_LFC'}),
            how='inner'
        ).dropna()
        
        corr = merged.corr().iloc[0, 1]
        print(f"LFC Pearson r (our vs GeneLab): {corr:.5f}")
        print(f"{'✅ Excellent' if corr > 0.99 else '⚠️  Check parameters'} (target: >0.99)")
        
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(merged['GL_LFC'], merged['log2FoldChange'],
                   alpha=0.25, s=5, color='steelblue', rasterized=True)
        lim = max(merged.abs().max().max(), 1) * 1.1
        ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1.5, label='y = x')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xlabel('GeneLab log2FC', fontsize=12)
        ax.set_ylabel('Our log2FC (PyDESeq2)', fontsize=12)
        ax.set_title(f'FLT vs GC: LFC Validation (r = {corr:.4f})', fontsize=12)
        ax.legend()
        plt.tight_layout()
        out_path = DIRS['counts'] / 'validation_LFC_correlation.png'
        plt.savefig(out_path, dpi=150)
        plt.show()
        print(f"✅ Saved: {out_path.name}")
    else:
        print("\n⚠️  Could not auto-detect LFC column.")
        print("   Manually set flt_gc_lfc to the correct column name from the list above.")


---
## Section 9: Over-Representation Analysis (ORA)

> **Overview:** Finding a list of DEGs is just the beginning.
> The real biology emerges when we ask: **do these genes cluster into known
> biological pathways or functions?**
>
> **Over-Representation Analysis (ORA)** answers: are the DEGs enriched for
> genes involved in specific pathways, compared to what we'd expect by chance?
>
> For example, if 500 genes are in your DEG list, and 50 of them are "immune
> system" genes, but only 100 out of 20,000 tested genes are "immune system"
> genes (0.5%), then 50/500 = 10% is massively over-represented. ORA uses a
> **hypergeometric test** (similar to Fisher's exact test) to calculate if
> this enrichment is statistically significant.
>
> **Gene sets used:**
> - **KEGG pathways**: curated metabolic and signaling pathways
> - **GO Biological Process**: Gene Ontology terms describing biological processes
>
> ORA only tests the list of significant DEGs (padj < 0.05, |LFC| > 1).


In [ ]:
def run_ora(res, contrast_name,
            gene_sets=['KEGG_2019_Mouse','GO_Biological_Process_2021'],
            lfc_thresh=1.0, padj_thresh=0.05):
    sig = res.dropna(subset=['padj'])
    sig = sig[(sig['padj'] < padj_thresh) & (sig['log2FoldChange'].abs() > lfc_thresh)]
    # Use gene name part (after underscore) if present, else use gene ID
    gene_list = [g.split('_')[-1] if '_' in g else g.split('.')[0]
                 for g in sig.index]
    gene_list = list(set(gene_list))   # deduplicate
    
    print(f"{contrast_name}: {len(gene_list)} DEGs submitted to ORA")
    if len(gene_list) < 5:
        print("  ⚠️  Too few genes.")
        return {}
    
    ora_results = {}
    for gs in gene_sets:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gs,
                organism='mouse',
                outdir=str(DIRS['counts'] / f"ORA_{contrast_name}_{gs[:10]}"),
                cutoff=0.05,
                no_plot=True
            )
            ora_results[gs] = enr.results
            top = enr.results[enr.results['Adjusted P-value'] < 0.05].head(5)
            print(f"  {gs}: {len(top)} significant terms")
            for _, row in top.head(3).iterrows():
                print(f"    {row['Term'][:55]}  (padj={row['Adjusted P-value']:.3e})")
        except Exception as e:
            print(f"  ⚠️  ORA failed for {gs}: {e}")
    return ora_results


ora_results = {}
for key, (c1, c2, label) in zip(results.keys(), CONTRASTS):
    print(f"\n{'='*55}")
    ora_results[key] = run_ora(results[key], key)


In [ ]:
def plot_ora_dotplot(ora_res, gs_name, contrast_name, top_n=15):
    if gs_name not in ora_res:
        return
    df = ora_res[gs_name][ora_res[gs_name]['Adjusted P-value'] < 0.05].head(top_n)
    if df.empty:
        print(f"No significant terms for {gs_name}")
        return
    df = df.copy()
    df['-log10(padj)'] = -np.log10(df['Adjusted P-value'].clip(lower=1e-30))
    df['Overlap_n']    = df['Overlap'].str.split('/').str[0].astype(int)
    df = df.sort_values('-log10(padj)', ascending=True)
    
    fig, ax = plt.subplots(figsize=(11, max(5, len(df) * 0.35)))
    sc = ax.scatter(df['-log10(padj)'], df['Term'],
                    s=df['Overlap_n'] * 6, c=df['-log10(padj)'],
                    cmap='viridis', alpha=0.85, edgecolors='white')
    ax.axvline(x=-np.log10(0.05), ls='--', lw=0.8, color='red', label='padj=0.05')
    ax.set_xlabel('-log10(adjusted p-value)', fontsize=11)
    ax.set_title(f"ORA: {gs_name[:25]}\n{contrast_name}",
                 fontsize=11, fontweight='bold')
    plt.colorbar(sc, ax=ax, label='-log10(padj)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    fname = DIRS['counts'] / f"ORA_dotplot_{contrast_name}_{gs_name[:10]}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {fname.name}")


for key in ['FLT_vs_GC', 'FLT_vs_VIV']:
    for gs in ['KEGG_2019_Mouse', 'GO_Biological_Process_2021']:
        plot_ora_dotplot(ora_results.get(key, {}), gs, key)


---
## Section 10: GSEA (Gene Set Enrichment Analysis)

> **Overview:** ORA only looks at the "significant" DEGs, but what
> about genes that show consistent trends without passing the strict significance
> threshold? **GSEA** solves this by using the **entire ranked gene list**.
>
> **How GSEA works:**
> 1. Rank all ~20,000 expressed genes by their log₂ fold change (most upregulated → most downregulated)
> 2. For each pathway, walk down the ranked list and calculate an enrichment score
>    (ES) that goes up when you hit a pathway gene and down when you don't
> 3. The maximum ES is the pathway's enrichment score; positive = pathway genes
>    are enriched among upregulated genes, negative = enriched among downregulated
> 4. Permutation testing gives a p-value and FDR
>
> **NES (Normalized Enrichment Score):** The ES normalized for pathway size.
> NES > 1.5 with FDR < 0.25 is typically considered significant in GSEA.
>
> GSEA is more sensitive than ORA for detecting subtle, coordinated changes
> across an entire pathway even when no single gene reaches significance.

Output: Normalized Enrichment Score (NES) and FDR q-value per pathway.


In [ ]:
def run_gsea(res, contrast_name,
             gene_sets=['KEGG_2019_Mouse','GO_Biological_Process_2021']):
    df  = res.dropna(subset=['log2FoldChange']).copy()
    rnk = df['log2FoldChange'].sort_values(ascending=False)
    rnk.index = [g.split('_')[-1] if '_' in g else g.split('.')[0]
                 for g in rnk.index]
    rnk = rnk[~rnk.index.duplicated()]
    
    print(f"{contrast_name}: {len(rnk):,} genes ranked")
    
    gsea_res = {}
    for gs in gene_sets:
        try:
            pre = gp.prerank(
                rnk=rnk, gene_sets=gs,
                threads=min(4, N_CORES),
                min_size=10, max_size=500,
                permutation_num=1000,
                outdir=str(DIRS['counts'] / f"GSEA_{contrast_name}_{gs[:10]}"),
                seed=42, verbose=False
            )
            gsea_res[gs] = pre
            top = pre.res2d[pre.res2d['FDR q-val'] < 0.25].head(5)
            print(f"  {gs}: {len(top)} terms FDR<0.25")
            for _, row in top.head(3).iterrows():
                print(f"    {row['Term'][:55]}  NES={float(row['NES']):.2f}  "
                      f"FDR={float(row['FDR q-val']):.3e}")
        except Exception as e:
            print(f"  ⚠️  GSEA failed for {gs}: {e}")
    return gsea_res


gsea_results = {}
for key, (c1, c2, label) in zip(results.keys(), CONTRASTS):
    print(f"\n{'='*55}")
    print(f"GSEA: {label}")
    gsea_results[key] = run_gsea(results[key], key)


In [ ]:
def plot_gsea_bar(gsea_res, gs_name, contrast_name, top_n=20):
    if gs_name not in gsea_res:
        return
    df = gsea_res[gs_name].res2d.copy()
    df['NES']       = df['NES'].astype(float)
    df['FDR q-val'] = df['FDR q-val'].astype(float)
    df = df[df['FDR q-val'] < 0.25]
    if df.empty:
        print(f"No GSEA terms (FDR<0.25) for {gs_name}")
        return
    df = df.reindex(df['NES'].abs().nlargest(top_n).index)
    df = df.sort_values('NES')
    
    norm   = plt.Normalize(df['FDR q-val'].min(), 0.25)
    colors = plt.cm.RdYlGn_r(norm(df['FDR q-val'].values))
    
    fig, ax = plt.subplots(figsize=(10, max(5, len(df) * 0.35)))
    ax.barh(df['Term'], df['NES'], color=colors, edgecolor='white', linewidth=0.3)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Normalized Enrichment Score (NES)', fontsize=11)
    ax.set_title(f"GSEA: {gs_name[:30]}\n{contrast_name} (FDR < 0.25)",
                 fontsize=11, fontweight='bold')
    sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=norm)
    plt.colorbar(sm, ax=ax, label='FDR q-value')
    plt.tight_layout()
    fname = DIRS['counts'] / f"GSEA_bar_{contrast_name}_{gs_name[:10]}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {fname.name}")


for key in ['FLT_vs_GC', 'FLT_vs_VIV']:
    for gs in ['KEGG_2019_Mouse', 'GO_Biological_Process_2021']:
        plot_gsea_bar(gsea_results.get(key, {}), gs, key)


---
## Section 11: Summary Table

> **Overview:** This final section produces a clean summary of
> how many DEGs were found in each comparison, and saves all results to an
> Excel workbook for easy sharing with collaborators who may not use Python.
>
> **Interpreting the final numbers:**
> - **FLT vs GC** DEGs: genes that change specifically due to the space environment
>   (gravity, radiation, etc.) beyond what's caused by the hardware itself
> - **FLT vs VIV** DEGs: total spaceflight effect vs normal housing conditions
> - **GC vs VIV** DEGs: effect of the special ground hardware (controls for FLT vs VIV)
>
> In a well-designed spaceflight study, GC vs VIV should have fewer DEGs than
> FLT vs GC, confirming that most differences are due to actual spaceflight, not
> the hardware.


In [ ]:
# DEG count summary
summary_rows = []
for key, (c1, c2, label) in zip(results.keys(), CONTRASTS):
    res = results[key].dropna(subset=['padj'])
    up  = ((res['padj'] < 0.05) & (res['log2FoldChange'] >  1)).sum()
    dn  = ((res['padj'] < 0.05) & (res['log2FoldChange'] < -1)).sum()
    summary_rows.append({
        'Contrast':       label,
        'Total DEGs':     up + dn,
        'Upregulated':    up,
        'Downregulated':  dn,
    })

summary_df = pd.DataFrame(summary_rows)
print("=" * 60)
print("DEG SUMMARY  (padj < 0.05, |log2FC| > 1)")
print("=" * 60)
display(summary_df)

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(summary_df))
w = 0.32
ax.bar(x - w/2, summary_df['Upregulated'],   width=w,
       color='#e74c3c', label='Upregulated')
ax.bar(x + w/2, summary_df['Downregulated'], width=w,
       color='#3498db', label='Downregulated')
ax.set_xticks(x)
ax.set_xticklabels([r['Contrast'] for r in summary_rows],
                   rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Number of DEGs', fontsize=11)
ax.set_title('GLDS-525 RR-23: DEG Counts per Contrast (padj<0.05, |LFC|>1)',
             fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
out_path = DIRS['counts'] / 'DEG_summary_barplot.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f"✅ Saved: {out_path.name}")


In [ ]:
# Save all DEG results to a single Excel workbook
excel_path = DIRS['counts'] / 'GLDS525_DEG_all_results.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    for key in results:
        sheet_name = key[:31]   # Excel sheet name limit
        results[key].to_excel(writer, sheet_name=sheet_name)
print(f"✅ All DEG results saved to: {excel_path.name}")

print("\n" + "="*60)
print("🎉 PIPELINE COMPLETE")
print("="*60)
print("\nOutput files in:", DIRS['counts'])
print("""
Key output files:
  DEGs_FLT_vs_GC_GLbulkRNAseq.csv         ← main spaceflight DEGs
  DEGs_FLT_vs_VIV_GLbulkRNAseq.csv
  DEGs_GC_vs_VIV_GLbulkRNAseq.csv
  GLDS525_DEG_all_results.xlsx             ← all results combined (Excel)
  volcano_plots_GLDS525.png                ← volcano plots
  PCA_GLDS525.png                          ← PCA plot
  heatmap_FLT_vs_GC.png                   ← top DEG heatmap
  ORA_*/                                   ← enrichr ORA results
  GSEA_*/                                  ← prerank GSEA results
""")
print("Congratulations on completing your first bulk RNA-seq analysis!")
